## Test DAGMC geometries of TPMS

In [ ]:
import openmc
import math
from openmc import volume
from matplotlib import pyplot as plt
print(openmc.__version__)
from warnings import warn
import openmc.lib
if not openmc.lib._dagmc_enabled():
    warn("DAGMC is not enabled.")
help(openmc.DAGMCUniverse.__init__)
import CoolProp.CoolProp as CP

0.15.2
Help on function __init__ in module openmc.dagmc:

__init__(
    self,
    filename: str | os.PathLike,
    universe_id=None,
    name='',
    auto_geom_ids=False,
    auto_mat_ids=False,
    material_overrides=None
)
    Initialize self.  See help(type(self)) for accurate signature.



In [13]:
# Remove existing OpenMC input XML files and output HDF5 files
!rm *.xml
!rm -f summary.h5
!rm -f statepoint.*.h5
!rm *.h5
!rm -f *.xml
# Reset ID
openmc.reset_auto_ids()

Generate the .h5m file tagging each .stl file with its material name:

In [ ]:
from stl_to_h5m import stl_to_h5m
stl_to_h5m(
    files_with_tags=[
        ("Water-pitch_16cm-por_73_33.stl", "Water"),
        ("Fuel-pitch_16cm-por_73_33.stl", "uo2"),
        ("Clad-pitch_16cm-por_73_33.stl", "Zircaloy"),
    ],
    h5m_filename="dagmc.h5m",
)

dagmc.h5m generated!


'\nfrom cad_to_h5m import cad_to_h5m\n\ncad_to_h5m(\n    files_with_tags=[\n        {"cad_filename": "water_cylinder.stl", "material_tag": "Water"},\n        {"cad_filename": "fuel_cylinder.stl",  "material_tag": "uo2"},\n        {"cad_filename": "clad_cylinder.stl",  "material_tag": "Zircaloy"},\n    ],\n    h5m_filename="dagmc.h5m",\n    cubit_path="/home/alessio_crosa/Coreform-Cubit-2025.12/bin",\n    make_watertight=False\n)\n'

Read from the imported .h5m file:

In [15]:
dagmc_universe = openmc.DAGMCUniverse(
    filename="dagmc.h5m",
    auto_geom_ids=True,
)

## Materials

In [ ]:
##################################################################################################################
################################ Materials for verification - Shriwise ###########################################
##### 2,4% enriched UO2
uo2 = openmc.Material(1, "uo2", 900.)
uo2.add_nuclide('U234', 4.4843e-06, 'ao')
uo2.add_nuclide('U235', 0.00055815, 'ao')
uo2.add_nuclide('U238', 0.022408, 'ao')
uo2.add_nuclide('O16', 0.045829, 'ao')
uo2.set_density('g/cm3', 10.29769)

#### Natural zirconium
clad= openmc.Material(2, "Zircaloy", 600.)
clad.add_nuclide('Zr90', 0.021827, 'ao')
clad.add_nuclide('Zr91', 0.00476, 'ao')
clad.add_nuclide('Zr92', 0.0072758, 'ao')
clad.add_nuclide('Zr94', 0.0073734, 'ao')
clad.add_nuclide('Zr96', 0.0011879, 'ao')
clad.set_density('g/cm3', 6.55)

###### Borated water (997 ppm)
water = openmc.Material(3, "Water", 600.)
water.add_nuclide('H1', 0.049457, 'ao')
water.add_nuclide('O16', 0.024672, 'ao')
water.add_nuclide('B10', 8.0042e-06, 'ao')
water.add_nuclide('B11', 3.2218e-05, 'ao')
water.set_density('g/cm3', 0.740582)
water.add_s_alpha_beta('c_H_in_H2O')

materials = openmc.Materials([uo2, clad, water])

'\n########################################### Materials for Ferney geometry ############################################################\n# FUEL: UO2 with 3,04% enriched Uranium\nuo2 = openmc.Material(1, "uo2", 900.)\nuo2.set_density(\'g/cm3\', 10.45)\nuo2.add_nuclide("U234", 6.15169E+18)\nuo2.add_nuclide("U235", 6.89220E+20)\nuo2.add_nuclide("U236", 3.16265E+18)\nuo2.add_nuclide("U238", 2.17103E+22)\nuo2.add_nuclide("C12", 9.13357E+18)\nuo2.add_nuclide("N14", 1.04072E+19)\nuo2.add_nuclide("O16", 4.48178E+22)\n\n\n# CLADDING: Zircaloy-4\nclad = openmc.Material(2, "Zircaloy", 600.)\nclad.set_density(\'g/cm3\', 6.56)\nclad.add_nuclide("O16", 1.19276E-03, percent_type="wo")\nclad.add_nuclide("O17", 4.82878E-07, percent_type="wo")\nclad.add_nuclide("O18", 2.75825E-06, percent_type="wo")\nclad.add_nuclide("Cr50", 4.16117E-05, percent_type="wo")\nclad.add_nuclide("Cr52", 8.34483E-04, percent_type="wo")\nclad.add_nuclide("Cr53", 9.64457E-05, percent_type="wo")\nclad.add_nuclide("Cr54", 2.446

In [17]:
# CROSS SECTION file indications
openmc.config['cross_sections'] = "/home/alessio_crosa/openmc_tpms/openmc-dev-alessio/openmc_xs/endfb-viii.0-hdf5/cross_sections.xml"

## Geometry and BCs

In [ ]:

############################ BOX for unit cell verification #######################################################
pitch = 16.0      # cm
# Cubic box with periodic BCs
xmin = openmc.XPlane(-pitch/2, boundary_type="periodic")
xmax = openmc.XPlane(+pitch/2, boundary_type="periodic")
ymin = openmc.YPlane(-pitch/2, boundary_type="periodic")
ymax = openmc.YPlane(+pitch/2, boundary_type="periodic")
zmin = openmc.ZPlane(-pitch/2, boundary_type="periodic")
zmax = openmc.ZPlane(+pitch/2, boundary_type="periodic")

# Link periodic faces in pairs
xmin.periodic_surface = xmax
ymin.periodic_surface = ymax
zmin.periodic_surface = zmax

# Box That include the tpms volumes (fuel, clad and water)
Box = openmc.Cell(name="root_cell")
Box.region = +xmin & -xmax & +ymin & -ymax & +zmin & -zmax
Box.fill = dagmc_universe

root_universe = openmc.Universe(cells=[Box])
geometry = openmc.Geometry(root_universe)

materials.export_to_xml()
geometry.export_to_xml()

'\n############################ CYLINDER for core verification  #######################################################\nradius = 10         # [cm]\nheight = 30         # [cm]\n\ntol = 5  # margine sicuro sopra l\'overshoot di 0.001\n\nlateral_surf = openmc.ZCylinder(r=radius)\nbottom_surf  = openmc.ZPlane(-(height/2), boundary_type="periodic")\nupper_surf   = openmc.ZPlane(+(height/2), boundary_type="periodic")\nbottom_surf.periodic_surface = upper_surf\n\n## For the reflector\nxP0 = openmc.XPlane(-(radius + tol), boundary_type="periodic")\nxP1 = openmc.XPlane(+(radius + tol), boundary_type="periodic")\nyP0 = openmc.YPlane(-(radius + tol), boundary_type="periodic")\nyP1 = openmc.YPlane(+(radius + tol), boundary_type="periodic")\nxP0.periodic_surface = xP1\nyP0.periodic_surface = yP1\nouter_box  = +xP0 & -xP1 & +yP0 & -yP1 & +bottom_surf & -upper_surf\nref_region = outer_box & +lateral_surf\n\n# Cell Core\ncell_core = openmc.Cell(name="core")\ncell_core.region = -lateral_surf & +bottom

In [ ]:
plot_xy = openmc.Plot()
plot_xy.basis = 'xy'
plot_xy.origin = (0, 0, 0)
plot_xy.width = (pitch, pitch)
plot_xy.pixels = (300, 300)
plot_xy.color_by = 'material'
#plot_xy.colors = {mFuel: 'red', mClad: 'grey', mCool: 'blue'}
openmc.plot_inline(plot_xy)

## Settings

In [ ]:
settings = openmc.Settings()
settings.temperature = {'method': 'interpolation'}
settings.batches = 200
settings.inactive = 15
settings.particles = 20000
settings.dagmc = True                         # explicit DAGMC

################################################# VOLUME calculation unit cell
lower_left = [-pitch/2, -pitch/2, -pitch/2]
upper_right = [pitch/2, pitch/2, pitch/2]
vol_calc = openmc.VolumeCalculation(
    domains=[water,uo2,clad], 
    samples=10_000_000, 
    lower_left=lower_left, 
    upper_right=upper_right
)
settings.volume_calculations = [vol_calc]
settings.export_to_xml()

# Execution of volume calculation
#openmc.calculate_volumes()

'\n################################################# VOLUME calculation CYLINDER\nlower_left  = [-radius, -radius, -height/2]\nupper_right  = [+radius, +radius, +height/2]\nvol_calc = openmc.VolumeCalculation(\n    domains=[water,uo2,clad], \n    samples=40_000_000, \n    lower_left=lower_left, \n    upper_right=upper_right\n)\nsettings.volume_calculations = [vol_calc]\nsettings.export_to_xml()\n'

## RUN OpenMC

openmc.run(output=True)